# 02 — Preprocessing Pipeline
**Project:** Explainable AI for IoT Intrusion Detection  
**Dataset:** CIC-BoT-IoT-V2  
**Model target:** Hybrid CNN-based classifier  

### What this notebook does (in order):
1. **Step 1** — Inspect & validate (infinities, duplicates, zero-variance columns)
2. **Step 2** — Drop irrelevant / leaky columns
3. **Step 3** — Feature engineering (Fwd/Bwd ratio, dispersion features)
4. **Step 4** — Log-transform skewed volumetric features
5. **Step 5** — Define the analyst-informed feature set
6. **Step 6** — Stratified train / val / test split (70 / 15 / 15)
7. **Step 7** — Fit StandardScaler on train only, transform all splits
8. **Step 8** — Handle class imbalance with SMOTE (train only)
9. **Step 9** — Reshape tensors for 1D-CNN input
10. **Step 10** — Save outputs (.npy arrays + scaler)

---
> **Install dependencies if needed:**
> ```
> pip install imbalanced-learn scikit-learn pandas numpy joblib
> ```

In [1]:
# ─────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib                          # For saving the scaler to disk
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

# Seed for reproducibility across all random operations
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports successful.')

All imports successful.


---
## Step 1 — Inspect & Validate
We check for three common issues in CIC datasets:
- **Infinity values** (`np.inf`, `-np.inf`) — common in rate features like `Flow Bytes/s`
- **Duplicate rows** — identical flows that could leak between train/test
- **Zero-variance columns** — features with a single unique value carry no information

In [2]:
# ─────────────────────────────────────────────────────────────────
# STEP 1A — Load data
# We assume df is already in memory from the EDA notebook.
# If not, uncomment the line below:
# df = pd.read_parquet('CIC-BoT-IoT-V2.parquet')
# ─────────────────────────────────────────────────────────────────

print(f'Dataset shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

NameError: name 'df' is not defined

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1B — Check for infinity values
#
# CIC flow exporters sometimes produce inf when dividing by zero
# (e.g. Flow Bytes/s when Flow Duration = 0). These must be
# replaced before scaling, or they will corrupt the entire column.
# ─────────────────────────────────────────────────────────────────

# Select only numeric columns for the inf check
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Count inf values per column (both +inf and -inf)
inf_counts = np.isinf(df[numeric_cols]).sum()
inf_cols = inf_counts[inf_counts > 0]

if len(inf_cols) > 0:
    print('Columns with infinity values:')
    print(inf_cols)
else:
    print('No infinity values found.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1C — Replace infinity values
#
# Strategy: replace inf with the column's median (not max), because
# max may itself be an inf or an extreme outlier.
# We use median because our EDA confirmed heavy right skew.
#
# NOTE: We do NOT clip or remove outlier spikes — our EDA showed
# that extreme high values in attack traffic ARE the signal.
# ─────────────────────────────────────────────────────────────────

# Work on a copy to preserve the original df from the EDA notebook
df_clean = df.copy()

# Replace inf/-inf with NaN first, then fill with column median
df_clean[numeric_cols] = df_clean[numeric_cols].replace([np.inf, -np.inf], np.nan)

# For each column that had inf values, fill NaN with the median
for col in numeric_cols:
    if df_clean[col].isna().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f'  Filled NaN in "{col}" with median = {median_val:.4f}')

# Verify no NaN or inf remain
assert not np.isinf(df_clean[numeric_cols].values).any(), 'Inf values still present!'
assert not df_clean[numeric_cols].isna().any().any(), 'NaN values still present!'
print('\nAll infinity and NaN values resolved.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1D — Check for and remove duplicate rows
#
# Duplicate rows (identical feature vectors AND label) are common
# in CIC datasets. If duplicates span train and test, the model
# effectively memorises test samples — inflating accuracy.
# ─────────────────────────────────────────────────────────────────

n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
n_after = len(df_clean)
n_dropped = n_before - n_after

print(f'Rows before deduplication : {n_before:,}')
print(f'Rows after  deduplication : {n_after:,}')
print(f'Duplicate rows removed    : {n_dropped:,} ({n_dropped/n_before*100:.2f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1E — Identify zero-variance columns
#
# A column with only one unique value has zero variance and provides
# no discriminative signal. From the EDA we already noticed
# 'Fwd Seg Size Min' was all zeros — this confirms it.
# ─────────────────────────────────────────────────────────────────

zero_var_cols = [col for col in numeric_cols 
                 if col not in ['Label'] and df_clean[col].nunique() == 1]

print(f'Zero-variance columns found: {zero_var_cols}')

# Also flag near-zero variance (std < 1e-6) for awareness
near_zero = [col for col in numeric_cols 
             if col not in ['Label'] and df_clean[col].std() < 1e-6 
             and col not in zero_var_cols]
if near_zero:
    print(f'Near-zero variance columns: {near_zero}')

---
## Step 2 — Drop Irrelevant / Leaky Columns

We remove:
- **`Attack`** — the string label column. `Label` (0/1) is our target. We save the Attack column separately for multi-class use later.
- **`Fwd Seg Size Min`** — confirmed zero-variance in EDA
- **Bulk features** — `Fwd/Bwd Avg Bytes/Bulk` etc. are almost always 0 in this dataset and add noise
- Any other zero-variance columns detected above

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — Drop columns that are non-informative or redundant
# ─────────────────────────────────────────────────────────────────

# Save the multi-class attack labels separately before dropping
# We will use these later for multi-class classification
attack_labels = df_clean['Attack'].copy()
print('Attack type distribution (saved for multi-class):')
print(attack_labels.value_counts())

# Columns to drop:
# - 'Attack': string category, not needed for binary classification
# - 'Fwd Seg Size Min': zero variance (all zeros)
# - Bulk rate features: near-zero in this dataset, add noise not signal
cols_to_drop = [
    'Attack',                  # String label — we use numeric 'Label'
    'Fwd Seg Size Min',        # Zero variance confirmed in EDA
    'Fwd Avg Bytes/Bulk',      # Near-zero in IoT traffic
    'Fwd Avg Packets/Bulk',    # Near-zero in IoT traffic
    'Fwd Avg Bulk Rate',       # Near-zero in IoT traffic
    'Bwd Avg Bytes/Bulk',      # Near-zero in IoT traffic
    'Bwd Avg Packets/Bulk',    # Near-zero in IoT traffic
    'Bwd Avg Bulk Rate',       # Near-zero in IoT traffic
]

# Also drop any additional zero-variance columns found in Step 1E
cols_to_drop += [c for c in zero_var_cols if c not in cols_to_drop]

# Only drop columns that actually exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in df_clean.columns]

df_clean = df_clean.drop(columns=cols_to_drop)

print(f'\nDropped {len(cols_to_drop)} columns: {cols_to_drop}')
print(f'Remaining columns: {df_clean.shape[1]}')

---
## Step 3 — Feature Engineering

Our EDA showed that **directional asymmetry** and **dispersion** are among the strongest class separators.  
The dataset already contains some dispersion features (`Packet Length Std`, `Packet Length Variance`),  
but we engineer two additional features that directly encode findings from the EDA:

| New Feature | Why | EDA Justification |
|---|---|---|
| `Fwd_Bwd_Pkt_Ratio` | Directional imbalance | Benign = balanced; Attack = one-sided |
| `Fwd_Bwd_Byte_Ratio` | Volume asymmetry | Same logic, on byte counts |

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — Engineer directional asymmetry features
#
# Both ratios use (fwd - bwd) / (fwd + bwd + 1) which:
#  - Returns 0 for perfectly balanced flows (benign IoT)
#  - Returns near +1/-1 for one-directional attack flows
#  - The +1 in the denominator avoids division by zero
#    when both fwd and bwd are 0
# ─────────────────────────────────────────────────────────────────

# Packet-level directional asymmetry
# High absolute values = strong directional imbalance = likely attack
df_clean['Fwd_Bwd_Pkt_Ratio'] = (
    (df_clean['Total Fwd Packets'] - df_clean['Total Backward Packets']) /
    (df_clean['Total Fwd Packets'] + df_clean['Total Backward Packets'] + 1)
)

# Byte-level directional asymmetry
# Exfiltration: high forward bytes. DoS flood: high forward packets.
df_clean['Fwd_Bwd_Byte_Ratio'] = (
    (df_clean['Fwd Packets Length Total'] - df_clean['Bwd Packets Length Total']) /
    (df_clean['Fwd Packets Length Total'] + df_clean['Bwd Packets Length Total'] + 1)
)

# Quick sanity check: print mean ratio by label
print('Mean Fwd_Bwd_Pkt_Ratio by class:')
print(df_clean.groupby('Label')['Fwd_Bwd_Pkt_Ratio'].mean())
print('\nMean Fwd_Bwd_Byte_Ratio by class:')
print(df_clean.groupby('Label')['Fwd_Bwd_Byte_Ratio'].mean())

print('\nNew engineered features added: Fwd_Bwd_Pkt_Ratio, Fwd_Bwd_Byte_Ratio')

---
## Step 4 — Log-Transform Skewed Volumetric Features

Our EDA confirmed **heavy right skew** in volumetric features (both classes).  
Log-transforming compresses the extreme scale differences so the CNN receives  
meaningful gradient signals across all features, not just the largest-valued ones.

We use `log1p(x) = log(1 + x)` which:
- Handles zero values safely (log(0) is undefined, but log(1+0) = 0)
- Preserves the relative ordering of values
- Is invertible with `expm1()` if we need to reverse it

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — Apply log1p transform to skewed volumetric features
#
# These are features confirmed to have heavy right skew in EDA.
# We do NOT transform:
#   - Flag counts (binary/small integers — skew is not a problem)
#   - Protocol (categorical)
#   - IAT features (their skew carries temporal signal — we keep raw)
#   - Already-engineered ratio features (bounded -1 to +1)
# ─────────────────────────────────────────────────────────────────

# Volumetric features with confirmed right skew from EDA
log_transform_cols = [
    # Volume / byte count features
    'Total Fwd Packets',
    'Total Backward Packets',
    'Fwd Packets Length Total',
    'Bwd Packets Length Total',
    'Fwd Packet Length Max',
    'Fwd Packet Length Mean',
    'Fwd Packet Length Std',       # Dispersion — keep but also log-transform
    'Bwd Packet Length Max',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Std',
    'Packet Length Max',
    'Packet Length Mean',
    'Packet Length Std',           # Key dispersion feature from EDA
    'Packet Length Variance',      # Key dispersion feature from EDA
    # Rate features
    'Flow Bytes/s',
    'Flow Packets/s',
    'Fwd Packets/s',
    'Bwd Packets/s',
    # Header lengths
    'Fwd Header Length',
    'Bwd Header Length',
    'Subflow Fwd Bytes',
    'Subflow Bwd Bytes',
]

# Only transform columns that actually exist after the drop step
log_transform_cols = [c for c in log_transform_cols if c in df_clean.columns]

# Apply log1p — safe for zero values, requires no negatives
# All volumetric features should be >= 0 in network flow data
for col in log_transform_cols:
    # Sanity check: ensure no negative values before transforming
    if df_clean[col].min() < 0:
        print(f'  WARNING: {col} has negative values — skipping log transform')
        continue
    df_clean[col] = np.log1p(df_clean[col])

print(f'Log1p transform applied to {len(log_transform_cols)} volumetric features.')

# Quick visual check: skew before vs after (on a sample)
# Compare 'Flow Bytes/s' distribution shape
print('\nSample feature stats after log transform (Flow Bytes/s):')
print(df_clean['Flow Bytes/s'].describe())

---
## Step 5 — Define the Analyst-Informed Feature Set

Based on the EDA findings and analyst domain knowledge, we select features across  
four groups ranked by their class separation strength:

| Group | Separation | Rationale |
|---|---|---|
| Temporal / IAT | **HIGH** | Distinguishes bursty attacks from periodic IoT |
| Directional | **HIGH** | Asymmetry is a strong behavioural attack indicator |
| Dispersion | **HIGH** | Variability beats magnitude for detecting irregularity |
| Volume / Rate | **MEDIUM** | Informative after log-transform |
| TCP Flags | **MEDIUM** | Captures connection setup/teardown anomalies |
| Protocol | **LOW** | Contextual only — kept for analyst interpretability |

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — Define the selected feature set
#
# This is the analyst-informed + EDA-validated selection.
# Grouping is preserved here for clarity and documentation.
# ─────────────────────────────────────────────────────────────────

# GROUP 1: Temporal / Inter-Arrival Time features
# These showed the STRONGEST class separation in EDA.
# They capture WHEN and HOW REGULARLY traffic arrives,
# distinguishing the periodic nature of benign IoT from attack bursts.
temporal_features = [
    'Flow Duration',
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Total',
    'Fwd IAT Mean',
    'Fwd IAT Std',
    'Fwd IAT Max',
    'Bwd IAT Total',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Active Mean',   # How long a flow was active before going idle
    'Idle Mean',     # How long a flow was idle — useful for C2 detection
]

# GROUP 2: Directional asymmetry features
# EDA showed CLEAR separation. Benign traffic is bidirectional;
# attack flows (DoS, DDoS, C2) show strong directional imbalance.
directional_features = [
    'Total Fwd Packets',
    'Total Backward Packets',
    'Fwd Packets Length Total',
    'Bwd Packets Length Total',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'Fwd_Bwd_Pkt_Ratio',    # Engineered in Step 3
    'Fwd_Bwd_Byte_Ratio',   # Engineered in Step 3
    'Down/Up Ratio',         # Similar concept, already in dataset
]

# GROUP 3: Dispersion / statistical variability features
# EDA KEY FINDING: Std outperforms Mean for separating classes.
# Irregular packet sizing is harder for attackers to mimic.
dispersion_features = [
    'Packet Length Std',       # Strongest dispersion signal in EDA
    'Packet Length Variance',  # Correlated but kept — CNN can learn
    'Fwd Packet Length Std',
    'Bwd Packet Length Std',
    'Flow IAT Std',            # Also in temporal — doubly important
]

# GROUP 4: Volume / rate features (post log-transform)
# Medium separation after log-transform. Captures flooding intensity.
volumetric_features = [
    'Flow Bytes/s',
    'Flow Packets/s',
    'Fwd Packet Length Max',
    'Bwd Packet Length Max',
    'Packet Length Max',
    'Packet Length Mean',
    'Avg Packet Size',
]

# GROUP 5: TCP flag / header features
# Protocol-level context for connection anomalies (SYN floods, RST storms)
flag_features = [
    'FIN Flag Count',
    'SYN Flag Count',
    'RST Flag Count',
    'PSH Flag Count',
    'ACK Flag Count',
    'Fwd PSH Flags',
    'Init Fwd Win Bytes',    # Window size — small values common in SYN floods
    'Init Bwd Win Bytes',
]

# GROUP 6: Protocol — complementary context (low discriminative power alone)
# SHAP is expected to rank this low, which itself is an informative finding.
context_features = [
    'Protocol',
]

# Combine all feature groups into the final selected set
# Use dict.fromkeys to remove any duplicates (e.g. Flow IAT Std appears twice)
selected_features = list(dict.fromkeys(
    temporal_features +
    directional_features +
    dispersion_features +
    volumetric_features +
    flag_features +
    context_features
))

# Keep only features that exist in the cleaned dataframe
selected_features = [f for f in selected_features if f in df_clean.columns]

print(f'Total features selected: {len(selected_features)}')
print('\nFeature groups:')
print(f'  Temporal       : {len([f for f in temporal_features if f in selected_features])}')
print(f'  Directional    : {len([f for f in directional_features if f in selected_features])}')
print(f'  Dispersion     : {len([f for f in dispersion_features if f in selected_features])}')
print(f'  Volumetric     : {len([f for f in volumetric_features if f in selected_features])}')
print(f'  Flags/Header   : {len([f for f in flag_features if f in selected_features])}')
print(f'  Context        : {len([f for f in context_features if f in selected_features])}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5B — Extract feature matrix X and label vector y
# ─────────────────────────────────────────────────────────────────

# X: feature matrix — only the selected analyst-informed features
X = df_clean[selected_features].values.astype(np.float32)

# y: binary label — 0 = Benign, 1 = Attack
y = df_clean['Label'].values.astype(np.int32)

print(f'X shape: {X.shape}  (samples x features)')
print(f'y shape: {y.shape}')
print(f'\nClass distribution:')
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    label_name = 'Benign' if u == 0 else 'Attack'
    print(f'  Class {u} ({label_name}): {c:,} ({c/len(y)*100:.2f}%)')

---
## Step 6 — Stratified Train / Validation / Test Split (70 / 15 / 15)

**Why stratified?**  
With ~99.7% attack samples, a random split might produce a val/test set  
with very few (or even zero) benign samples. Stratification guarantees  
the class ratio is preserved in every split.

**Two-step approach:**  
Scikit-learn's `train_test_split` doesn't directly support 3-way splits,  
so we do it in two steps: first split off 70% train, then split the  
remaining 30% equally into val and test.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 6 — Stratified 70 / 15 / 15 split
# ─────────────────────────────────────────────────────────────────

# Step 6a: Split 70% train vs 30% temp (which will become val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,            # 30% held out for val + test
    stratify=y,                # Preserve class proportions in each split
    random_state=RANDOM_STATE
)

# Step 6b: Split the 30% temp into 15% val and 15% test
# test_size=0.5 of 30% = 15% of total
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,            # Half of 30% = 15% of total each
    stratify=y_temp,
    random_state=RANDOM_STATE
)

# Report split sizes and class distributions
total = len(y)
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    n = len(y_split)
    n_benign = (y_split == 0).sum()
    n_attack = (y_split == 1).sum()
    print(f'{name:6s}: {n:>8,} rows ({n/total*100:.1f}%)  '
          f'| Benign: {n_benign:,} ({n_benign/n*100:.2f}%)  '
          f'| Attack: {n_attack:,} ({n_attack/n*100:.2f}%)')

print(f'\nTotal: {total:,} rows')

---
## Step 7 — StandardScaler (Fit on Train Only)

**Critical rule:** The scaler must be **fit on training data only**.  
Fitting on the full dataset leaks test set statistics into the model — a form  
of data leakage that inflates evaluation metrics.

After fitting, we apply the same scaler (same mean/std) to val and test.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 7 — Fit StandardScaler on TRAINING SET ONLY
#
# StandardScaler subtracts the mean and divides by std deviation,
# giving each feature zero mean and unit variance.
# This is critical for CNNs to converge smoothly.
# ─────────────────────────────────────────────────────────────────

scaler = StandardScaler()

# FIT on training data only — this learns the mean and std of each feature
X_train_scaled = scaler.fit_transform(X_train)

# TRANSFORM val and test using the training statistics
# NEVER fit again on val/test — that would be leakage
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Verify the scaling looks correct on training data
# After StandardScaler: mean ≈ 0, std ≈ 1 for each feature
print('Post-scaling stats on training set (should be near 0 mean, 1 std):')
print(f'  Mean across all features: {X_train_scaled.mean(axis=0).mean():.6f}')
print(f'  Std  across all features: {X_train_scaled.std(axis=0).mean():.6f}')
print('\nNote: Val/Test stats will not be exactly 0/1 — that is expected and correct.')
print(f'  Val mean: {X_val_scaled.mean(axis=0).mean():.6f}')
print(f'  Test mean: {X_test_scaled.mean(axis=0).mean():.6f}')

---
## Step 8 — SMOTE: Handle Class Imbalance (Train Only)

Our dataset has **~99.7% attack, ~0.3% benign** — an extreme imbalance.  
Without correction, the model can achieve 99.7% accuracy by predicting  
everything as attack, while completely missing benign traffic.

**SMOTE** (Synthetic Minority Over-sampling Technique) generates synthetic  
benign samples by interpolating between real benign samples in feature space.

**Rules:**
- Apply SMOTE **after** splitting — never on the full dataset
- Apply SMOTE **after** scaling — synthetic samples should be in scaled space
- Apply **only to the training set** — val/test must reflect real-world distribution

> ⚠️ **Note on 11.5M rows:** SMOTE on this scale will use significant RAM.  
> If memory is constrained, use `sampling_strategy=0.1` (10% ratio) instead  
> of full 50/50 balance, or use `class_weight='balanced'` in your Keras model.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 8A — Check class distribution BEFORE SMOTE
# ─────────────────────────────────────────────────────────────────

unique, counts = np.unique(y_train, return_counts=True)
print('Training class distribution BEFORE SMOTE:')
for u, c in zip(unique, counts):
    print(f'  Class {u}: {c:,} ({c/len(y_train)*100:.3f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 8B — Apply SMOTE to TRAINING SET ONLY
#
# sampling_strategy=0.2 means: oversample minority (benign) until
# it represents 20% of the majority class count.
# This is a pragmatic compromise — full 50/50 on 11M rows is very
# memory intensive. Adjust as needed.
#
# If you want full balance, set sampling_strategy=1.0
# If memory is tight, set sampling_strategy=0.1
# ─────────────────────────────────────────────────────────────────

smote = SMOTE(
    sampling_strategy=0.2,    # Benign will be 20% of attack count
    k_neighbors=5,            # Standard SMOTE neighbourhood size
    random_state=RANDOM_STATE,
    n_jobs=-1                 # Use all CPU cores — important for large datasets
)

print('Applying SMOTE to training set... (may take a few minutes on large data)')
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Verify new distribution
unique, counts = np.unique(y_train_resampled, return_counts=True)
print('\nTraining class distribution AFTER SMOTE:')
for u, c in zip(unique, counts):
    print(f'  Class {u}: {c:,} ({c/len(y_train_resampled)*100:.2f}%)')

print(f'\nTraining set size: {len(y_train):,} → {len(y_train_resampled):,}')

---
## Step 9 — Reshape for 1D-CNN Input

Keras 1D-CNN (`Conv1D`) expects input shape:  
`(num_samples, num_timesteps, num_channels)` 

For tabular data treated as a feature sequence:  
- `num_timesteps` = number of features  
- `num_channels` = 1 (each feature is one channel)  

So we reshape `(N, F)` → `(N, F, 1)`.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 9 — Reshape to (samples, features, 1) for 1D-CNN
#
# The last dimension '1' is the channel dimension.
# Conv1D will treat each feature as a position in a sequence
# and learn local feature interactions (e.g. between adjacent
# temporal features in the ordered feature set).
# ─────────────────────────────────────────────────────────────────

def reshape_for_cnn(X):
    """Reshape 2D array (samples, features) to 3D (samples, features, 1)
    as required by Keras Conv1D layers."""
    return X.reshape(X.shape[0], X.shape[1], 1)

# Reshape resampled training set (post-SMOTE)
X_train_cnn = reshape_for_cnn(X_train_resampled)

# Reshape val and test (NOT resampled — keep real-world distribution)
X_val_cnn   = reshape_for_cnn(X_val_scaled)
X_test_cnn  = reshape_for_cnn(X_test_scaled)

print('Final tensor shapes for CNN:')
print(f'  X_train_cnn : {X_train_cnn.shape}  (train, post-SMOTE)')
print(f'  X_val_cnn   : {X_val_cnn.shape}')
print(f'  X_test_cnn  : {X_test_cnn.shape}')
print(f'  y_train     : {y_train_resampled.shape}')
print(f'  y_val       : {y_val.shape}')
print(f'  y_test      : {y_test.shape}')
print(f'\nInput shape for CNN model: ({X_train_cnn.shape[1]}, {X_train_cnn.shape[2]})')
print(f'  i.e. {X_train_cnn.shape[1]} features, 1 channel')

---
## Step 10 — Save Preprocessed Outputs

We save:
- All CNN-ready arrays as `.npy` — fast to reload, no re-running preprocessing
- The fitted `StandardScaler` as `.pkl` — **must be used** when the dashboard receives new traffic for inference
- The feature name list as `.npy` — needed by SHAP to label feature importance plots

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 10 — Save all outputs
#
# File inventory:
#   X_train_cnn.npy  — Training features (post-SMOTE, scaled, reshaped)
#   X_val_cnn.npy    — Validation features (scaled, reshaped)
#   X_test_cnn.npy   — Test features (scaled, reshaped)
#   y_train.npy      — Training labels (post-SMOTE)
#   y_val.npy        — Validation labels
#   y_test.npy       — Test labels
#   scaler.pkl       — Fitted StandardScaler (use for dashboard inference)
#   feature_names.npy — Ordered list of feature names (for SHAP plots)
# ─────────────────────────────────────────────────────────────────

import os

# Create output directory if it doesn't exist
output_dir = 'preprocessed_data'
os.makedirs(output_dir, exist_ok=True)

# Save numpy arrays
np.save(f'{output_dir}/X_train_cnn.npy', X_train_cnn)
np.save(f'{output_dir}/X_val_cnn.npy',   X_val_cnn)
np.save(f'{output_dir}/X_test_cnn.npy',  X_test_cnn)
np.save(f'{output_dir}/y_train.npy',     y_train_resampled)
np.save(f'{output_dir}/y_val.npy',       y_val)
np.save(f'{output_dir}/y_test.npy',      y_test)

# Save feature names as numpy array of strings
# This is required by SHAP to produce labelled importance plots
np.save(f'{output_dir}/feature_names.npy', np.array(selected_features))

# Save the fitted scaler — CRITICAL for dashboard inference
# Any new traffic sample must pass through this scaler before prediction
joblib.dump(scaler, f'{output_dir}/scaler.pkl')

print('All preprocessed data saved to:', output_dir)
print('\nFiles saved:')
for fname in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<30s}  {size_mb:.1f} MB')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 10B — Final preprocessing summary
# ─────────────────────────────────────────────────────────────────

print('=' * 60)
print('PREPROCESSING COMPLETE — SUMMARY')
print('=' * 60)
print(f'\nOriginal dataset      : {df.shape[0]:>12,} rows × {df.shape[1]} cols')
print(f'After deduplication   : {len(df_clean)+len(y_train_resampled)-len(y_train):>12,} rows (approx)')
print(f'Features selected     : {len(selected_features):>12}')
print(f'  of which engineered : {2:>12}  (Fwd_Bwd_Pkt_Ratio, Fwd_Bwd_Byte_Ratio)')
print(f'\nSplit sizes:')
print(f'  Train (post-SMOTE)  : {len(y_train_resampled):>12,}')
print(f'  Validation          : {len(y_val):>12,}')
print(f'  Test                : {len(y_test):>12,}')
print(f'\nCNN input shape       : ({X_train_cnn.shape[1]}, {X_train_cnn.shape[2]})')
print(f'\nOutputs saved to      : ./{output_dir}/')
print('\nNext step: 03_model_training.ipynb')
print('  → Build hybrid CNN architecture')
print('  → Model input_shape =', (X_train_cnn.shape[1], 1))
print('  → Load data with: np.load("preprocessed_data/X_train_cnn.npy")')

---
## How to Load This Data in the Next Notebook

```python
import numpy as np
import joblib

# Load preprocessed arrays
X_train = np.load('preprocessed_data/X_train_cnn.npy')
X_val   = np.load('preprocessed_data/X_val_cnn.npy')
X_test  = np.load('preprocessed_data/X_test_cnn.npy')
y_train = np.load('preprocessed_data/y_train.npy')
y_val   = np.load('preprocessed_data/y_val.npy')
y_test  = np.load('preprocessed_data/y_test.npy')

# Load feature names (needed for SHAP plots)
feature_names = np.load('preprocessed_data/feature_names.npy', allow_pickle=True).tolist()

# Load scaler (needed for dashboard inference)
scaler = joblib.load('preprocessed_data/scaler.pkl')

# CNN input shape
input_shape = (X_train.shape[1], X_train.shape[2])  # (n_features, 1)
```